# Offline (Batch) RL: learning a policy from logged data alone

_Reinforcement Learning_

**You can't try anything new — only a fixed log of past experience. The danger: trusting Q on actions the data never tried.**

Everything so far in this curriculum assumed an agent that acts: value iteration
       (rl-value-iteration) knew the model and swept it; Q-learning (ai-q-learning) and
       DQN (mod-dqn, Deep Q-Network) collected their own experience, trying actions and
       letting the observed rewards correct their estimates. Offline RL removes the acting. You are handed
       a frozen log $\mathcal{D} = \{(s_i, a_i, r_i, s'_i)\}$ and must produce the best policy you can, then
       deploy it &mdash; never touching the environment during learning.

       That single change breaks the algorithms you know. To see why, recall the Q-learning target
       (rl-bellman-optimality): $r + \gamma \max_{a'} Q(s', a')$. That $\max$ over actions is the
       trap. Off-line, the network will happily report a huge $Q(s', a')$ for some action $a'$ that was
       never logged in $s'$ &mdash; an out-of-distribution (OOD) action. The $\max$ seeks out
       exactly the most over-estimated action. Online, you would take $a'$, get the real (disappointing) reward,
       and the update would pull $Q(s',a')$ back down. Offline, there is no such feedback: nothing ever
       contradicts the inflated value, it propagates into every Bellman backup, and $Q$ diverges.

---

This notebook is a practice scaffold for the **Offline (Batch) RL: learning a policy from logged data alone** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — numpy (tabular) — d3rlpy for the deep version

In [ ]:
# Offline (batch) RL, tabular numpy: naive Q-learning OVERESTIMATES on unseen
# (state, action) pairs and DIVERGES; a conservative variant stays stable.
# No environment interaction after the dataset is collected -- that is offline RL.
import numpy as np
rng = np.random.default_rng(0)

# --- A tiny environment: a 5-state chain. States 0..4; state 4 is terminal (goal).
# Two actions: 0 = "left" (stay/step back), 1 = "right" (advance toward goal).
# Reward +1 only on entering the terminal goal; 0 otherwise. gamma = 0.9.
N_STATES, N_ACTIONS, GAMMA = 5, 2, 0.9
GOAL = N_STATES - 1

def step(s, a):
    s_next = max(0, s - 1) if a == 0 else min(GOAL, s + 1)
    r = 1.0 if s_next == GOAL and s != GOAL else 0.0
    return s_next, r

# --- 1) COLLECT A FIXED DATASET from a BEHAVIOUR policy that is biased:
# it almost always goes RIGHT (action 1). So action 0 ("left") is RARELY logged
# in most states -> those (s, a=0) pairs are OUT-OF-DISTRIBUTION (OOD).
def behaviour_action(s):
    return 1 if rng.random() < 0.9 else 0     # 90% right, 10% left

dataset = []                                   # frozen log of (s, a, r, s')
s = 0
for _ in range(4000):
    if s == GOAL:
        s = 0                                  # episode reset
        continue
    a = behaviour_action(s)
    s_next, r = step(s, a)
    dataset.append((s, a, r, s_next))
    s = s_next
dataset = np.array(dataset, dtype=float)

# Count coverage: which (s, a) pairs did the log actually see?
counts = np.zeros((N_STATES, N_ACTIONS))
for s_, a_, _, _ in dataset:
    counts[int(s_), int(a_)] += 1
print("dataset (s,a) visitation counts:\n", counts.astype(int))
# The a=0 column is tiny -> those actions are barely/never supported.

def offline_q_learning(conservative, n_sweeps=60, alpha=5.0, lr=0.5):
    """Fitted Q-iteration over the FIXED dataset. If conservative, penalise Q on
    (s,a) pairs the dataset under-covers (tabular stand-in for CQL)."""
    Q = np.zeros((N_STATES, N_ACTIONS))
    history = []                               # max |Q| per sweep, to watch blow-up
    for _ in range(n_sweeps):
        for s_, a_, r_, sp_ in dataset:
            s_, a_, sp_ = int(s_), int(a_), int(sp_)
            target = r_ + GAMMA * Q[sp_].max() # the dangerous max over ALL actions
            Q[s_, a_] += lr * (target - Q[s_, a_])
        if conservative:
            # CQL-style: push Q DOWN on under-supported (OOD) actions, leave
            # well-covered ones alone. Penalty scales with how unseen the pair is.
            for s_ in range(N_STATES):
                for a_ in range(N_ACTIONS):
                    support = counts[s_, a_] / (counts[s_].sum() + 1e-9)
                    if support < 0.2:          # OOD / under-supported action
                        Q[s_, a_] -= alpha * (0.2 - support)
        history.append(np.abs(Q).max())
    return Q, history

Q_naive, hist_naive = offline_q_learning(conservative=False)
Q_cons,  hist_cons  = offline_q_learning(conservative=True)

print("\nnaive   max|Q| over sweeps:", [round(h, 1) for h in hist_naive[::10]])
print("conserv max|Q| over sweeps:", [round(h, 1) for h in hist_cons[::10]])
# naive   max|Q| ... grows without bound  -> 10s, 100s, ...   (DIVERGES)
# conserv max|Q| ... settles near 1/(1-gamma)=10              (STABLE)

# Greedy policy each method commits to (action per state):
print("\nnaive   greedy policy:", Q_naive.argmax(1))
print("conserv greedy policy:", Q_cons.argmax(1))
# Conservative -> all 'right' (action 1): the SUPPORTED, goal-reaching choice.

# ---------------------------------------------------------------------------
# Deep / scaled version with d3rlpy (Colab: !pip install d3rlpy):
#
# import d3rlpy
# from d3rlpy.algos import CQLConfig, DQNConfig
# dataset, env = d3rlpy.datasets.get_dataset("hopper-medium-v0")  # fixed D4RL log
# # Naive offline DQN -> Q overestimates, poor return:
# DQNConfig().create(device="cuda:0").fit(dataset, n_steps=100000)
# # Conservative CQL -> stable Q, strong return on the SAME fixed dataset:
# CQLConfig(alpha=1.0).create(device="cuda:0").fit(dataset, n_steps=100000)
# d3rlpy also ships IQL, BCQ, BEAR, and Decision Transformer (DiscreteDecisionTransformer).

## Visualize the data & results

_On the SAME fixed offline dataset, what happens to the learned Q-values (and the resulting return) when naive offline Q-learning trusts the max over all actions, versus a conservative method that suppresses out-of-distribution actions?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 5-state chain, state 4 terminal/goal; a=0 left, a=1 right; +1 on entering goal.
N_STATES, N_ACTIONS, GAMMA, GOAL = 5, 2, 0.9, 4
def step(s, a):
    s_next = max(0, s-1) if a == 0 else min(GOAL, s+1)
    r = 1.0 if (s_next == GOAL and s != GOAL) else 0.0
    return s_next, r

# --- Fixed dataset from a RIGHT-BIASED behaviour policy (a=0 rarely logged -> OOD).
data, s = [], 0
for _ in range(4000):
    if s == GOAL: s = 0; continue
    a = 1 if rng.random() < 0.9 else 0
    sp, r = step(s, a); data.append((s, a, r, sp)); s = sp
data = np.array(data, float)
counts = np.zeros((N_STATES, N_ACTIONS))
for s_, a_, _, _ in data: counts[int(s_), int(a_)] += 1

def fit(conservative, sweeps=60, alpha=5.0, lr=0.5, ood_bias=8.0):
    Q = np.zeros((N_STATES, N_ACTIONS)); hist = []
    for k in range(sweeps):
        # Inject the worst-case extrapolation a function approximator gives an
        # UNSEEN action: a large optimistic value the max will then chase.
        if not conservative:
            for s_ in range(N_STATES):
                for a_ in range(N_ACTIONS):
                    if counts[s_, a_] / (counts[s_].sum()+1e-9) < 0.2:
                        Q[s_, a_] = max(Q[s_, a_], ood_bias)   # phantom high value
        for s_, a_, r_, sp_ in data:
            s_, a_, sp_ = int(s_), int(a_), int(sp_)
            Q[s_, a_] += lr * (r_ + GAMMA*Q[sp_].max() - Q[s_, a_])
        if conservative:                       # CQL-style: push OOD actions DOWN
            for s_ in range(N_STATES):
                for a_ in range(N_ACTIONS):
                    sup = counts[s_, a_] / (counts[s_].sum()+1e-9)
                    if sup < 0.2: Q[s_, a_] -= alpha * (0.2 - sup)
        hist.append(np.abs(Q).max())
    return Q, hist

Q_naive, h_naive = fit(False)
Q_cons,  h_cons  = fit(True)
print("naive   max|Q| trace:", [round(x,1) for x in h_naive[::5]])
print("conserv max|Q| trace:", [round(x,1) for x in h_cons[::5]])

# --- Roll out each greedy policy on the TRUE env; average discounted return.
def evaluate(policy, episodes=2000, horizon=20):
    tot = 0.0
    for _ in range(episodes):
        s, G, disc = 0, 0.0, 1.0
        for _ in range(horizon):
            if s == GOAL: break
            s, r = step(s, policy(s)); G += disc*r; disc *= GAMMA
        tot += G
    return tot / episodes

beh = evaluate(lambda s: 1 if rng.random() < 0.9 else 0)
nai = evaluate(lambda s: int(Q_naive[s].argmax()))
con = evaluate(lambda s: int(Q_cons[s].argmax()))
print(f"return  behaviour={beh:.3f}  naive={nai:.3f}  conservative={con:.3f}")
# return  behaviour=0.478  naive=0.000  conservative=0.656
# naive max|Q| diverges (47+ and climbing); conservative settles near 10.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In one sentence, why does naive offline Q-learning OVERESTIMATE the value of out-of-distribution (OOD) actions, and why &mdash; unlike online Q-learning &mdash; does the error never get corrected?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the culprit term: the Bellman target $r + \gamma\max_{a'} Q(s',a')$. — _The $\max$ deliberately selects the action with the largest $Q$._
- Note that off-support $Q$ is an unchecked extrapolation, often inflated, so the $\max$ tends to pick exactly those inflated OOD values (maximisation bias, Jensen's inequality). — _$\mathbb{E}[\max_{a'} Q] \ge \max_{a'} \mathbb{E}[Q]$: the expected max is biased upward._
- Observe there is no environment interaction offline, so the inflated value is never contradicted by a real reward. — _Online, you would take the action and the observed reward would pull $Q$ back down; offline that loop is severed._

**Answer:** The greedy $\max$ in the backup seeks out the highest $Q$, which for unseen (OOD) actions is an over-estimate from extrapolation (maximisation bias); offline there is no chance to try that action and observe its true, lower reward, so the inflated value is never corrected, propagates through every Bellman backup, and $Q$ diverges.

</details>

**Problem 2.** CQL (Conservative Q-Learning) adds the penalty $\alpha\bigl(\mathbb{E}_{a\sim\pi}[Q(s,a)] - \mathbb{E}_{(s,a)\sim\mathcal{D}}[Q(s,a)]\bigr)$ to the Bellman loss. Explain what each of the two expectations does, and what happens if $\alpha$ is too large.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the first term $\mathbb{E}_{a\sim\pi}[Q]$: averaged over actions the current policy would take; we MINIMISE the loss, so this term PUSHES $Q$ DOWN there. — _Those policy actions are the ones that might wander off-support; pushing them down removes the phantom high values the $\max$ would chase._
- Read the second term $-\mathbb{E}_{\mathcal{D}}[Q]$: averaged over actions actually in the data; the minus sign PULLS $Q$ UP there. — _We must stay accurate on what we genuinely observed, not collapse everything._
- Consider $\alpha$ very large. — _The penalty dominates the Bellman fit, so $Q$ becomes over-pessimistic: even good but rare in-data actions get suppressed, and the policy under-performs._

**Answer:** The first expectation lowers $Q$ on the policy's (possibly OOD) actions; the second raises $Q$ on the actions present in the data &mdash; together making $Q$ a lower bound off-support while staying honest on-support, so the $\max$ can't be fooled. If $\alpha$ is too large the penalty overwhelms the Bellman fit: $Q$ turns over-pessimistic, suppressing even good rare actions and yielding a timid, sub-optimal policy.

</details>

**Problem 3.** Off-policy evaluation by importance sampling reweights a logged trajectory's return by $\prod_{t=0}^{H}\rho_t$ with $\rho_t = \pi(a_t\mid s_t)/\pi_\beta(a_t\mid s_t)$. The estimator is unbiased &mdash; so why is it often useless in practice for long horizons?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the weight is a PRODUCT of $H{+}1$ ratios, one per step. — _Each ratio compares how likely the target vs behaviour policy was to take the logged action._
- Reason about the product's spread: if any step has $\pi_\beta$ small, that ratio is huge; if $\pi$ small, near zero; the product multiplies these swings. — _Multiplying $H{+}1$ noisy factors makes variance grow roughly exponentially in the horizon $H$._
- Conclude that a few trajectories with enormous weights dominate the average. — _Unbiased in expectation, but any finite sample is dominated by rare high-weight trajectories &mdash; the estimate is wildly noisy._

**Answer:** Because the weight is a product of per-step ratios, its variance grows roughly exponentially with the horizon $H$: a handful of trajectories acquire gigantic weights and dominate every finite-sample estimate. It is unbiased but high-variance &mdash; hence weighted IS, per-decision IS, and doubly-robust estimators that trade a little bias for far less variance.

</details>